In [4]:
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

### Parsing Contaminantes

In [5]:
anios = range(2014, 2026)
contaminantes_dict = {}
contaminantes_ids = {
    2014: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-29-calidad-aire-diario-csv/download/201410-29-calidad-aire-diario-csv.csv",
    2015: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-27-calidad-aire-diario-csv/download/201410-27-calidad-aire-diario-csv.csv",
    2016: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-26-calidad-aire-diario-csv/download/201410-26-calidad-aire-diario-csv.csv",
    2017: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-25-calidad-aire-diario-csv/download/201410-25-calidad-aire-diario-csv.csv",
    2018: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-20-calidad-aire-diario-csv/download/201410-20-calidad-aire-diario-csv.csv",
    2019: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-17-calidad-aire-diario-csv/download/201410-17-calidad-aire-diario-csv.csv",
    2020: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-14-calidad-aire-diario-csv/download/201410-14-calidad-aire-diario-csv.csv",
    2021: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-10-calidad-aire-diario-csv/download/201410-10-calidad-aire-diario-csv.csv",
    2022: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-72-calidad-aire-diario-csv/download/201410-72-calidad-aire-diario-csv.csv",
    2023: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-5-calidad-aire-diario-csv/download/201410-5-calidad-aire-diario-csv.csv",
    2024: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-4-calidad-aire-diario-csv/download/201410-4-calidad-aire-diario-csv.csv",
    2025: "https://datos.madrid.es/dataset/201410-0-calidad-aire-diario/resource/201410-2-calidad-aire-diario-csv/download/201410-2-calidad-aire-diario-csv.csv"
}

polen_dict = {}

for anio in anios:
    try:
        file_cont = contaminantes_ids[anio]
        contaminantes_dict[anio] = pd.read_csv(file_cont, sep=';')
        print(f"Año {anio} cargado correctamente.")
        
    except Exception as e:
        print(f"Error al cargar el año {anio}: {e}")

Año 2014 cargado correctamente.
Año 2015 cargado correctamente.
Año 2016 cargado correctamente.
Año 2017 cargado correctamente.
Año 2018 cargado correctamente.
Año 2019 cargado correctamente.
Año 2020 cargado correctamente.
Año 2021 cargado correctamente.
Año 2022 cargado correctamente.
Año 2023 cargado correctamente.
Año 2024 cargado correctamente.
Año 2025 cargado correctamente.


In [6]:
df_contaminantes_total = pd.concat(contaminantes_dict.values(), ignore_index=True)

df_contaminantes_total = df_contaminantes_total[(df_contaminantes_total['MUNICIPIO'] == 79) & 
                 (df_contaminantes_total['ESTACION'] == 8)].copy()

magnitude_map = {
    1: 'SO2 (ug/m3)', 6: 'CO (mg/m3)', 7: 'NO (ug/m3)', 8: 'NO2 (ug/m3)',
    9: 'PM2.5 (ug/m3)', 10: 'PM10 (ug/m3)', 12: 'NOx (ug/m3)', 14: 'O3 (ug/m3)',
    20: 'Tolueno (ug/m3)', 30: 'Benceno (ug/m3)', 42: 'HCT (mg/m3)', 44: 'HCNM (mg/m3)'
}

d_cols = [f'D{i:02d}' for i in range(1, 32)]
v_cols = [f'V{i:02d}' for i in range(1, 32)]

rows = []
for idx, row in df_contaminantes_total.iterrows():
    for d, v in zip(d_cols, v_cols):
        day_num = int(d[1:])
        if row[v] == 'V':
            val = str(row[d]).replace(',', '.')
            rows.append({
                'ano': row['ANO'], 'mes': row['MES'], 'dia': day_num,
                'magnitud': row['MAGNITUD'],
                'valor': pd.to_numeric(val, errors='coerce')
            })

df_melted = pd.DataFrame(rows)

# 5. Crear la columna de fecha y limpiar errores (ej. 31 de febrero)
df_melted['fecha'] = pd.to_datetime(df_melted[['ano', 'mes', 'dia']].rename(
    columns={'ano': 'year', 'mes': 'month', 'dia': 'day'}), errors='coerce')
df_melted = df_melted.dropna(subset=['fecha'])

# 6. Pivotar: Magnitudes a columnas
df_melted['magnitud_nombre'] = df_melted['magnitud'].map(magnitude_map)
df_final_contaminantes = df_melted.pivot_table(index='fecha', columns='magnitud_nombre', values='valor')

df_final_contaminantes = df_final_contaminantes.reset_index()
df_final_contaminantes = df_final_contaminantes.sort_values('fecha')

df_final_contaminantes = df_final_contaminantes.drop(columns=["HCNM (mg/m3)", "HCT (mg/m3)"])

df_final_contaminantes.to_csv(r"..\my_datasets/contaminantes_2014_2025.csv", index=False, encoding='utf-8')

df_final_contaminantes

magnitud_nombre,fecha,Benceno (ug/m3),CO (mg/m3),NO (ug/m3),NO2 (ug/m3),NOx (ug/m3),O3 (ug/m3),PM10 (ug/m3),PM2.5 (ug/m3),SO2 (ug/m3),Tolueno (ug/m3)
0,2014-01-01,0.5,0.2,16.0,35.0,60.0,27.0,9.0,7.0,6.0,0.8
1,2014-01-02,0.5,0.2,35.0,45.0,98.0,26.0,9.0,7.0,6.0,1.6
2,2014-01-03,0.5,0.2,36.0,49.0,104.0,26.0,12.0,8.0,4.0,2.4
3,2014-01-04,0.4,0.1,14.0,32.0,53.0,46.0,8.0,5.0,4.0,0.9
4,2014-01-05,0.5,0.2,16.0,36.0,61.0,38.0,14.0,8.0,4.0,1.1
...,...,...,...,...,...,...,...,...,...,...,...
4432,2026-04-26,0.2,0.2,0.0,8.0,8.0,78.0,18.0,8.0,3.0,1.1
4433,2026-04-27,0.3,0.3,1.0,16.0,17.0,76.0,22.0,9.0,3.0,2.3
4434,2026-04-28,0.3,0.2,0.0,13.0,14.0,91.0,21.0,7.0,3.0,1.7
4435,2026-04-29,0.2,0.3,0.0,12.0,12.0,70.0,21.0,9.0,3.0,1.5


### Parsing OPEN-METEO

In [7]:
URL_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"

params_meteo = {
    "latitude": 40.4165,
    "longitude": -3.7026,
    "start_date": "2014-01-01",
    "end_date": "2025-12-31",
    "hourly": [
        "wind_speed_10m",
        "wind_gusts_10m",
        "relative_humidity_2m",
        "dew_point_2m",
        "vapour_pressure_deficit",
        "cloud_cover",
        "soil_temperature_0_to_7cm",
        "soil_moisture_0_to_7cm",
    ],
    "daily": [
        "temperature_2m_mean",
        "wind_direction_10m_dominant",
        "et0_fao_evapotranspiration",
        "shortwave_radiation_sum",
        "temperature_2m_max",
        "temperature_2m_min",
        "rain_sum"
    ],
    "timezone": "Europe/Madrid"
}
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)
responses = openmeteo.weather_api(URL_METEO_ARCHIVE, params=params_meteo)
response = responses[0]

hourly = response.Hourly()
hourly_data = {
    "date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    ),
    "wind_speed_10m (km/h)": hourly.Variables(0).ValuesAsNumpy(),
    "wind_gusts_10m (km/h)": hourly.Variables(1).ValuesAsNumpy(),
    "relative_humidity_2m (%)": hourly.Variables(2).ValuesAsNumpy(),
    "dew_point_2m (°C)": hourly.Variables(3).ValuesAsNumpy(),
    "vapour_pressure_deficit (kPa)": hourly.Variables(4).ValuesAsNumpy(),
    "cloud_cover (%)": hourly.Variables(5).ValuesAsNumpy(),
    "soil_temperature_0_to_7cm (°C)": hourly.Variables(6).ValuesAsNumpy(),
    "soil_moisture_0_to_7cm (m³/m³)": hourly.Variables(7).ValuesAsNumpy()
}
hourly_dataframe = pd.DataFrame(data = hourly_data)

daily = response.Daily()
daily_data = {
    "date": pd.date_range(
        start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
        end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = daily.Interval()),
        inclusive = "left"
    ),
    "temp_mean": daily.Variables(0).ValuesAsNumpy(),
    "wind_dir_dominant": daily.Variables(1).ValuesAsNumpy(),
    "evapotranspiration": daily.Variables(2).ValuesAsNumpy(),
    "radiation_sum": daily.Variables(3).ValuesAsNumpy(),
    "max_temp": daily.Variables(4).ValuesAsNumpy(),
    "min_temp": daily.Variables(5).ValuesAsNumpy(),
    "total_rain": daily.Variables(6).ValuesAsNumpy()
}
df_diario = pd.DataFrame(data = daily_data)

hourly_dataframe['fecha'] = pd.to_datetime(hourly_dataframe['date']).dt.date
df_meteo_pred = hourly_dataframe.groupby('fecha').mean(numeric_only=True).reset_index()
df_meteo_pred['fecha'] = pd.to_datetime(df_meteo_pred['fecha']).dt.tz_localize(None)
df_diario['date'] = pd.to_datetime(df_diario['date']).dt.tz_localize(None).dt.normalize()
df_final = pd.merge(df_meteo_pred, df_diario, left_on='fecha', right_on='date', how='left')
df_meteo_final = df_final.drop(columns=['date'])

df_meteo_final.to_csv(r"..\my_datasets/meteo_2014_2025.csv", index=False, encoding='utf-8')

### Parsing Polen + MIX (polen + contaminantes + meteo)

In [8]:
años = range(2014, 2026)
tipos_polinicos = {
    "Gram": "gramineas",
    "Cupres": "cupresaceas",
    "Olivo": "olivo",
    "Pl": "platano",
    "Urtic": "urticaceas",
    "Queno": "quenopodiaceas"
}
resource_ids = {
    2014: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/d5f64a9a-de2a-45e0-9a67-02894025fb8b/download/2014.csv",
    2015: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/24e8fb64-dfa3-4088-bc43-3cacd2063544/download/2015.csv",
    2016: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/8ab1db74-0c04-4615-9e43-8ae2972cee10/download/2016.csv",
    2017: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/6965c4ed-c1a8-45a0-a9cc-e51e1e2403b8/download/2017.csv",
    2018: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/4a944c0a-e316-4d4b-b12b-05c51c65691d/download/2018.csv",
    2019: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/51179a09-694c-4579-88b4-4ebc451da603/download/2019.csv",
    2020: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/a86b8045-f455-4cbc-9523-8fb7020dbfa6/download/2020.csv",
    2021: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/2615b31b-a74a-4674-89c2-85cf63ee7f42/download/2021.csv",
    2022: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/fed85c1e-c5e8-49cf-9985-a39fb10f978c/download/2022.csv",
    2023: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/adaaa9ec-196f-417d-80dc-0f9ee1813b3a/download/2023.csv",
    2024: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/58005206-9020-4c4e-8cd0-16089760f02f/download/2024.csv",
    2025: "https://datos.comunidad.madrid/catalogo/dataset/b0f1e9eb-dfe6-4baf-8167-4fa7a0710509/resource/fc197e69-c3fb-4cea-a763-4609d9a66fad/download/2025.csv"
}

polen_dict = {}

for año, res_id in resource_ids.items():
    try:
        file_polen = resource_ids[año]
        
        df_polen = pd.read_csv(file_polen, encoding='latin1', sep=';')
        df_polen = df_polen[df_polen['captador'] == 'AYTM'].copy()
        df_polen['fecha'] = pd.to_datetime(df_polen['fecha_lectura']).dt.date
        df_polen.drop(columns=['fecha_lectura', 'captador'], inplace=True)
        
        polen_dict[año] = df_polen
        print(f"Año {año} cargado correctamente.")
        
    except Exception as e:
        print(f"Error al cargar el año {año}: {e}")

Año 2014 cargado correctamente.
Año 2015 cargado correctamente.
Año 2016 cargado correctamente.
Año 2017 cargado correctamente.
Año 2018 cargado correctamente.
Año 2019 cargado correctamente.
Año 2020 cargado correctamente.
Año 2021 cargado correctamente.
Año 2022 cargado correctamente.
Año 2023 cargado correctamente.
Año 2024 cargado correctamente.
Año 2025 cargado correctamente.


In [9]:
for inicales, nombre_limpio in tipos_polinicos.items():
    lista_dfs = []
    for año in años:
        df = polen_dict[año]
        filtro = df[df['tipo_polinico'].str.contains(inicales, na=False, case=False)].copy()
        filtro['año'] = año
        lista_dfs.append(filtro)
    df_polen_total = pd.concat(lista_dfs, ignore_index=True)

    df_polen_total['fecha'] = pd.to_datetime(df_polen_total['fecha']).dt.normalize()
    df_polen_total = df_polen_total.groupby('fecha')['granos_de_polen_x_metro_cubico'].mean().reset_index()
    df_polen_total = df_polen_total.rename(columns={'granos_de_polen_x_metro_cubico': f'polen_{nombre_limpio}'})
    
    df_meteo_final['fecha'] = pd.to_datetime(df_meteo_final['fecha']).dt.normalize()
    df_final_contaminantes['fecha'] = pd.to_datetime(df_final_contaminantes['fecha']).dt.normalize()

    df_WMeteo = pd.merge(df_polen_total, df_meteo_final, on='fecha', how='outer')
    df_WMeteo_WContaminantes = pd.merge(df_WMeteo, df_final_contaminantes, on='fecha', how='outer')
    
    df_WMeteo_WContaminantes = df_WMeteo_WContaminantes.sort_values('fecha')

    df_WMeteo_WContaminantes.to_csv(rf"..\my_datasets\datos_{nombre_limpio}.csv", index=False)